# Bước 7 — Đánh giá mô hình bổ trợ

Nhóm 10, IE313. Vận dụng Bài 06–07; nguồn/trang PDF theo `docs/SPECIFICATION.md` mục 1.3–1.4. Quy ước và phần bổ sung tại `docs/MODELING_METHODS.md`. Notebook đọc baseline giữ nguyên giá dương, chọn biến trên train/fold, xuất kết quả thực; không hạ ngưỡng để ép có hồi quy.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "workspaces/final-project"]
PROJECT = next(p for p in candidates if (p / "src/modeling.py").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.modeling import run_full_modeling_workflow  # noqa: E402

metadata = run_full_modeling_workflow(
    PROJECT / "data/processed/bangkok_cleaned.csv", PROJECT / "reports/modeling"
)
print(metadata["versions"])

[*] 1/4 Đang phân chia dữ liệu Train (70%) / Test (30%) và thiết lập KFold (4 folds)...


[✓] Đã xuất bản bảng gán tập: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/reports/modeling/split_assignments.csv
[*] 2/4 Đang thực hiện chọn lọc đặc trưng độc lập trong từng training fold...


[✓] Đã xuất bản kết quả chọn đặc trưng: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/reports/modeling/feature_selection.json
[*] 3/4 Đang đánh giá mốc cơ sở DummyRegressor (y_train mean benchmark)...
[*] 4/4 Đang tổng hợp bảng so sánh mô hình và siêu dữ liệu...
[✓] Đã xuất bản bảng so sánh mô hình: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/reports/modeling/model_comparison.csv
[✓] Đã xuất bản siêu dữ liệu mô hình: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/reports/modeling/modeling_metadata.json
[✓] Đã xuất bản báo cáo phát hiện mô hình: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/reports/modeling/modeling_findings.md
{'python': '3.12.3', 'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'scikit-learn': '1.9.1'}


## Chia tập và tính độc lập

70/30 theo listing, seed42; CV 4-fold trên train. ID/host không được làm predictor. Giao host giới hạn suy rộng sang chủ nhà mới. Mỗi fold có quyết định chọn biến riêng trước khi fit imputer/scaler/encoder.

In [2]:
display(pd.Series(metadata["split_info"]))
assignments = pd.read_csv(PROJECT / "reports/modeling/split_assignments.csv")
display(assignments.groupby(["split", "fold"]).size().rename("n"))

n_total                                                        28987
n_train                                                        20290
n_test                                                          8697
test_size_ratio                                                  0.3
random_state                                                      42
n_cv_splits                                                        4
host_overlap       {'total_hosts_train': 5879, 'total_hosts_test'...
dtype: object

split  fold
test   -1      8697
train   0      5073
        1      5073
        2      5072
        3      5072
Name: n, dtype: int64

## Chọn biến số và chẩn đoán phân loại

Pearson |r|≥0,30 và p<0,05 là quy ước repository; thống kê chưa làm tròn. **[MỞ RỘNG NGOÀI BÀI HỌC]** Levene center=median chỉ chẩn đoán phương sai, nguồn trong EDA_METHODS. Không gán mọi giả định là sai khi chưa đo; nhóm n=1 được giữ và ghi rõ ANOVA chưa tính. Chính sách thận trọng không dùng ANOVA làm bộ lọc khi điều kiện chưa được xác nhận.

In [3]:
selection = json.loads((PROJECT / "reports/modeling/feature_selection.json").read_text())
display(pd.DataFrame(selection["full_train"]["numeric_features"]).T)
display(
    pd.DataFrame(selection["full_train"]["categorical_features"]).T[
        ["groups_count", "n_valid", "f_statistic", "p_value", "selected", "reason"]
    ]
)
print("Số biến qua tiêu chí từng fold:", selection["folds_selected_counts"])
print("Số biến full train:", selection["full_train_selected_count"])

,n_valid,pearson_r,p_value,selected,reason
accommodates,20290,0.103597,0.0,False,CORRELATION_TOO_WEAK (|r|=0.1036 < 0.3)
bedrooms,17780,0.088128,0.0,False,CORRELATION_TOO_WEAK (|r|=0.0881 < 0.3)
bathrooms,18353,0.08759,0.0,False,CORRELATION_TOO_WEAK (|r|=0.0876 < 0.3)
beds,19386,0.069044,0.0,False,CORRELATION_TOO_WEAK (|r|=0.0690 < 0.3)
minimum_nights,20289,-0.022791,0.001168,False,CORRELATION_TOO_WEAK (|r|=0.0228 < 0.3)
number_of_reviews,20290,0.001082,0.877473,False,CORRELATION_TOO_WEAK (|r|=0.0011 < 0.3)
review_scores_rating,13702,-0.024696,0.00384,False,CORRELATION_TOO_WEAK (|r|=0.0247 < 0.3)


,groups_count,n_valid,f_statistic,p_value,selected,reason
room_type,4,20290,3.702228,0.011177,False,ANOVA_ASSUMPTIONS_NOT_CONFIRMED_FOR_SELECTION
property_type,71,20290,None,None,False,ANOVA_ASSUMPTIONS_NOT_CONFIRMED_FOR_SELECTION
neighbourhood_cleansed,50,20290,1.273393,0.095025,False,ANOVA_ASSUMPTIONS_NOT_CONFIRMED_FOR_SELECTION
host_is_superhost,2,20290,5.602068,0.017949,False,ANOVA_ASSUMPTIONS_NOT_CONFIRMED_FOR_SELECTION


Số biến qua tiêu chí từng fold: [0, 0, 0, 0]
Số biến full train: 0


## Kết quả đánh giá

DummyRegressor là **bổ sung kỹ thuật theo repository**, dự đoán trung bình y_train riêng từng fold. Nó là mốc so sánh, không thay thế hồi quy có predictor. MSE theo THB²; R² giữ giá trị âm, trả null khi không xác định. CV std dùng ddof=0. Khi có biến hợp lệ, lựa chọn cấu hình bằng CV và chỉ đánh giá holdout sau khi chốt.

In [4]:
display(pd.DataFrame([metadata["benchmark_results"]]))
display(pd.read_csv(PROJECT / "reports/modeling/model_comparison.csv"))
print(metadata["feasibility_decision"])
print(metadata["conditional_artifacts"])

,model_name,train_mean_prediction_thb,train_mse,train_r2,train_r2_reason,test_mse,test_r2,test_r2_reason,cv_4fold_mse_mean,cv_4fold_mse_std,cv_4fold_mse_by_fold,cv_4fold_r2_mean,cv_4fold_r2_std,cv_4fold_r2_by_fold,cv_r2_reasons,cv_n_splits,cv_std_ddof,status
0,DummyRegressor (Mean Benchmark - Bổ sung kỹ th...,2976.121534,3.836658e+08,0.0,None,3.150212e+08,-0.000014,None,3.837150e+08,2.189993e+08,"[58292575.94176567, 538236364.4512585, 6232151...",-0.000535,0.000606,"[-0.0015508545904099336, -0.000430325845862000...","[None, None, None, None]",4,0,COMPLETED_BENCHMARK


,model_name,model_type,features_used,train_mse,train_r2,cv_4fold_mse_mean,cv_4fold_r2_mean,test_mse,test_r2,status,notes
0,DummyRegressor (Mean Benchmark - Bổ sung kỹ th...,Benchmark,None (y_train mean),3.836658e+08,0.0,3.837150e+08,-0.000535,3.150212e+08,-0.000014,COMPLETED_BENCHMARK,Mốc cơ sở tham chiếu (Baseline Comparator); li...
1,Simple Linear Regression (SLR - Bài 06),SLR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT_FEASIBLE_NO_SELECTED_NUMERIC_FEATURES,"Theo quy tắc |r|≥0.3 và p<0.05, số biến chọn ở..."
2,Multiple Linear Regression (MLR - Bài 06),MLR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT_FEASIBLE_NO_SELECTED_FEATURES,"Theo quy tắc |r|≥0.3 và p<0.05, số biến chọn ở..."
3,Polynomial Regression (Degree 2 - Bài 06),Poly-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT_FEASIBLE_NO_SELECTED_NUMERIC_FEATURES,"Theo quy tắc |r|≥0.3 và p<0.05, số biến chọn ở..."
4,Ridge Regression (alpha tuning - Bài 07),Ridge,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT_FEASIBLE_NO_SELECTED_FEATURES,"Theo quy tắc |r|≥0.3 và p<0.05, số biến chọn ở..."


{'is_modeling_feasible': False, 'infeasibility_reason': 'Theo quy tắc |r|≥0.3 và p<0.05, số biến chọn ở các fold là [0, 0, 0, 0]; full train có 0 biến. Fold không có biến: fold_1, fold_2, fold_3, fold_4. Biến phân loại chưa được chọn do giả định dùng ANOVA chưa xác nhận; xem chẩn đoán từng biến/fold. Chưa triển khai hồi quy theo chính sách hiện tại; đây không phải chứng minh dữ liệu không thể mô hình hóa.'}
{'best_pipeline_joblib': None, 'model_diagnostics_npz': None, 'diagnostic_figures': None, 'status_explanation': 'Chưa có hồi quy theo quy tắc chọn biến hiện tại; chỉ có mốc trung bình.', 'archived_previous_artifacts': []}


## Tổng hợp và giới hạn

Diễn giải theo đúng kết quả đang chạy. Chưa chọn được biến theo tiêu chí hiện tại không chứng minh dữ liệu không thể mô hình hóa. Khi không có hồi quy, không xuất file mô hình hay hình chẩn đoán hồi quy. Nhánh phần mềm khả thi được kiểm thử trên dữ liệu tổng hợp; không dùng số liệu kiểm thử trong kết quả đồ án.

In [5]:
display(Markdown((PROJECT / "reports/modeling/modeling_findings.md").read_text()))

# Kết quả Bước 7 sau review

Nguồn: bangkok_cleaned.csv; SHA-256 `961a01ad225e35f989ec81f124925488e392baca2ef3800ff4028b4423d5f9a7`.
Train 20,290, test 8,697; seed 42; CV 4 fold.

## Chọn biến trong train/fold

Ngưỡng số: |r|≥0,30 và p<0,05 theo repository. Biến phân loại chưa được chọn khi chưa có cơ sở dùng ANOVA làm bộ lọc. Đây là chính sách phân tích, không phải định lý về khả năng dự báo.

| Biến số | n train | r train | p train | Được chọn |
|---|---:|---:|---:|---|
| accommodates | 20290 | 0.10359685 | 1.5648382e-49 | False |
| bedrooms | 17780 | 0.088127676 | 5.3665267e-32 | False |
| bathrooms | 18353 | 0.087589847 | 1.3643475e-32 | False |
| beds | 19386 | 0.069044172 | 6.3236294e-22 | False |
| minimum_nights | 20289 | -0.022790891 | 0.0011681544 | False |
| number_of_reviews | 20290 | 0.001082419 | 0.87747292 | False |
| review_scores_rating | 13702 | -0.024696398 | 0.0038398455 | False |

Số biến từng fold: [0, 0, 0, 0]; full train: 0.

| Biến phân loại | Số nhóm | F thăm dò | p thăm dò | Trạng thái |
|---|---:|---:|---:|---|
| room_type | 4 | 3.7022279 | 0.01117702 | COMPUTED |
| property_type | 71 | Không xác định | Không xác định | NOT_COMPUTABLE |
| neighbourhood_cleansed | 50 | 1.2733929 | 0.095025106 | COMPUTED |
| host_is_superhost | 2 | 5.6020681 | 0.017948623 | COMPUTED |

Các nhóm n=1 không bị âm thầm loại để tính lại ANOVA. N, skew, Levene và host trong từng training fold lưu tại `feature_selection.json`. **[MỞ RỘNG NGOÀI BÀI HỌC]** Levene center=median; nguồn/giới hạn tại `docs/EDA_METHODS.md`. Skew và mất cân đối chỉ là chẩn đoán, không tự chứng minh giả định đúng/sai.

## Quyết định theo quy tắc hiện tại

Theo quy tắc |r|≥0.3 và p<0.05, số biến chọn ở các fold là [0, 0, 0, 0]; full train có 0 biến. Fold không có biến: fold_1, fold_2, fold_3, fold_4. Biến phân loại chưa được chọn do giả định dùng ANOVA chưa xác nhận; xem chẩn đoán từng biến/fold. Chưa triển khai hồi quy theo chính sách hiện tại; đây không phải chứng minh dữ liệu không thể mô hình hóa.

## Mốc dự đoán trung bình

Giá dự đoán cố định = 2976.1215 THB/đêm. DummyRegressor là API bổ sung kỹ thuật; không thay thế hồi quy có predictor.

| Thước đo | Train | CV mean ± std | Test |
|---|---:|---:|---:|
| MSE (THB²) | 3.8366581e+08 | 3.8371497e+08 ± 2.189993e+08 | 3.150212e+08 |
| R² | 0 | -0.00053479395 ± 0.00060631197 | -1.4334218e-05 |

R² = 1 − SSE/SST với trung bình của chính tập đang đánh giá trong SST; mẫu hằng/ít hơn 2 trả null. CV std dùng ddof=0; không bỏ fold chưa xác định. Số đầy đủ trong JSON/CSV, chỉ làm tròn khi trình bày.

## Giới hạn và sản phẩm

2,229 host giao hai tập; 7,202 listing test thuộc host đã thấy (82.81%). 82.81% listing trong tập test thuộc về các chủ nhà đã xuất hiện trong tập train. Phân chia ngẫu nhiên theo listing chỉ đánh giá khả năng dự báo trên các chỗ ở mới trong cùng tập dữ liệu, KHÔNG đánh giá khả năng tổng quát hóa sang chủ nhà mới.
Các p-value mang tính thăm dò. Không suy ra tác động nhân quả, tính độc lập hay khả năng tổng quát sang host mới.
Sản phẩm hồi quy hiện hành: `{'best_pipeline_joblib': None, 'model_diagnostics_npz': None, 'diagnostic_figures': None, 'status_explanation': 'Chưa có hồi quy theo quy tắc chọn biến hiện tại; chỉ có mốc trung bình.', 'archived_previous_artifacts': []}`.
Bảng so sánh và trạng thái: `model_comparison.csv`; metadata có phiên bản môi trường. Phương pháp: `docs/MODELING_METHODS.md`.
